# 3.1. Implementar un sistema STRIPS básico para proporcionar movimiento a un robot

Partiendo de las nociones teóricas y prácticas vistas en la unidad, el alumno va a diseñar y desarrollar un sistema STRIPS para simular el movimiento de un robot. Tanto el diseño como el código deben ser entregados describiendo detalladamente cada uno de los procesos involucrados, así como la justificación y pertinencia de su implementación.

# Diseño del sistema

## ¿Qué es STRIPS?

STRIPS (*Stanford Research Institute Problem Solver*) es el lenguaje de representación de problemas de planificación propuesto por Fikes y Nilsson (1971) y el origen de la planificación clásica en IA. Un problema STRIPS se define con tres elementos:

- **Estado**: una colección de hechos que son verdaderos en un momento dado (p. ej. `en(robot, cocina)`). Todo lo que no aparece listado se asume falso — es la *hipótesis de mundo cerrado*.
- **Operadores (acciones)**: cada uno tiene una **lista de precondiciones** (hechos que deben cumplirse para poder ejecutarlo), una **lista-añadir** (hechos que se vuelven verdaderos al ejecutarlo) y una **lista-borrar** (hechos que dejan de serlo).
- **Meta**: un conjunto de hechos que se quiere lograr; el problema se resuelve encontrando una secuencia de operadores (un **plan**) que, aplicados desde el estado inicial, produzcan un estado donde la meta se cumple.

Esta práctica implementa ese modelo de forma directa: los estados son conjuntos de tuplas, los operadores son objetos con sus tres listas, y el planificador es una búsqueda en el espacio de estados (Russell & Norvig, 2020, cap. 11) que aplica operadores hasta alcanzar la meta.

## Dominio de aplicación: un robot doméstico que se desplaza entre habitaciones

El área elegida es la navegación de un **robot doméstico** dentro de una casa con varias habitaciones conectadas por puertas, algunas de las cuales pueden estar cerradas. Además de moverse, el robot puede **tomar y dejar objetos**, para poder plantear una tarea completa y realista: llevar un objeto de una habitación a otra. Es un dominio clásico para ilustrar STRIPS porque combina varias acciones interdependientes (moverse depende de que la puerta esté abierta; tomar un objeto depende de estar en la habitación correcta) sin la complejidad de un dominio industrial.

## Mapa de la casa

| Habitación | Conectada con | Puerta | Estado inicial de la puerta |
|---|---|---|---|
| cocina | pasillo | `p1` | abierta |
| pasillo | sala | `p2` | cerrada |
| pasillo | comedor | `p3` | abierta |
| sala | recámara | `p4` | cerrada |

Estado inicial: el robot está en la **cocina**, con las manos libres, y hay una **caja** en el **comedor**. La tarea principal que se plantea más adelante es que el robot **lleve la caja hasta la recámara**, lo que lo obliga a atravesar varias habitaciones y a abrir las dos puertas cerradas en el camino.

In [9]:
import sys
from collections import deque
from dataclasses import dataclass

print(f"Python {sys.version}")

Python 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


## Representación del estado y de las acciones

Un **hecho** es una tupla, p. ej. `("en", "robot", "cocina")` o `("abierta", "p1")`. Un **estado** es un `frozenset` de hechos (inmutable, para poder guardarlo como visitado durante la búsqueda). Una **acción** es un objeto `Accion` con sus tres listas STRIPS:

- `precondiciones`: hechos que deben estar en el estado actual para poder ejecutar la acción.
- `add_list`: hechos que se agregan al estado al ejecutarla.
- `del_list`: hechos que se eliminan del estado al ejecutarla.

`aplicable` comprueba que `precondiciones` sea subconjunto del estado; `aplicar` calcula el estado resultante como `(estado - del_list) | add_list`, la operación estándar de actualización de un operador STRIPS.

In [10]:
@dataclass(frozen=True)
class Accion:
    nombre: str
    precondiciones: frozenset
    add_list: frozenset
    del_list: frozenset

    def aplicable(self, estado: frozenset) -> bool:
        return self.precondiciones <= estado

    def aplicar(self, estado: frozenset) -> frozenset:
        return (estado - self.del_list) | self.add_list

## ¿Por qué `frozenset` y no `set`?

Los estados (`estado_inicial`, cada estado sucesor) y `meta` son todos del mismo tipo de dato: un **`frozenset` de tuplas**, donde cada tupla es un hecho (p. ej. `("en", "robot", "cocina")`). Hay dos razones concretas para usar `frozenset` en vez de `set`, no es una preferencia de estilo:

**1. El planificador necesita meter estados dentro de un `set` de visitados.** Para que un objeto pueda vivir dentro de un `set` (o ser clave de un `dict`) Python necesita poder calcularle un **hash**, y solo los objetos **inmutables** son hasheables — si el contenido pudiera cambiar después de insertado, el objeto quedaría en el "cajón" equivocado de la tabla hash. Un `set` normal es mutable (`.add()`, `.remove()`), así que **no** es hasheable; un `frozenset` sí, porque una vez creado no se puede modificar.

**2. `Accion` es un `@dataclass(frozen=True)`**, y eso exige que **todos** sus campos sean hasheables para que el propio objeto `Accion` lo sea. Si `precondiciones`, `add_list` o `del_list` fueran `set` normales, la clase dejaría de ser coherente con su propio contrato de inmutabilidad.

Un `frozenset` ofrece exactamente las mismas operaciones de conjuntos que un `set` (`|` unión, `-` diferencia, `<=` subconjunto), que son justo las que usan `aplicable` y `aplicar` — la diferencia es únicamente que no se puede modificar después de creado. La celda siguiente lo comprueba de forma directa.

In [11]:
# Demostración: un estado como set normal no se puede usar como visitado
estado_mutable = {("en", "robot", "cocina")}
try:
    hash(estado_mutable)
except TypeError as error:
    print(f"hash(set) falla -> {error}")

estado_inmutable = frozenset(estado_mutable)
print(f"hash(frozenset) sí funciona -> {hash(estado_inmutable)}")

visitados_demo = {estado_inmutable}
print(f"¿el frozenset cabe en un set de visitados? {estado_inmutable in visitados_demo}")

hash(set) falla -> unhashable type: 'set'
hash(frozenset) sí funciona -> 396121421673777810
¿el frozenset cabe en un set de visitados? True


## Operadores del dominio (esquemas STRIPS)

Se definen cuatro esquemas de acción. Cada función recibe los parámetros concretos (habitaciones, puertas, objetos) e instancia un `Accion` con sus listas ya resueltas para esos parámetros — es la fase de **grounding** (instanciación) de la planificación clásica.

| Esquema | Precondiciones | Lista-añadir | Lista-borrar |
|---|---|---|---|
| `Mover(desde, hasta, puerta)` | robot en `desde`; `puerta` conecta `desde` con `hasta`; `puerta` abierta | robot en `hasta` | robot en `desde` |
| `Abrir(puerta, habitación)` | robot en `habitación`; `puerta` cerrada | `puerta` abierta | `puerta` cerrada |
| `Tomar(objeto, habitación)` | robot en `habitación`; `objeto` en `habitación`; manos libres | robot sosteniendo `objeto` | `objeto` en `habitación`; manos libres |
| `Dejar(objeto, habitación)` | robot en `habitación`; robot sosteniendo `objeto` | `objeto` en `habitación`; manos libres | robot sosteniendo `objeto` |

Nótese que mientras el robot sostiene un objeto, este **no tiene una habitación asociada** en el estado (se eliminó al tomarlo): así, cuando el robot se mueve, el objeto se mueve implícitamente con él sin necesitar una acción adicional, y solo vuelve a tener una ubicación fija cuando se ejecuta `Dejar`.

`generar_operadores` recorre la lista de conexiones del mapa y la lista de objetos para instanciar automáticamente **todos** los operadores del dominio (en ambos sentidos de cada puerta), en vez de escribirlos uno por uno a mano.

In [12]:
def accion_mover(desde: str, hasta: str, puerta: str) -> Accion:
    return Accion(
        nombre=f"Mover(robot, {desde} -> {hasta}, por {puerta})",
        precondiciones=frozenset({("en", "robot", desde), ("conecta", puerta, desde, hasta), ("abierta", puerta)}),
        add_list=frozenset({("en", "robot", hasta)}),
        del_list=frozenset({("en", "robot", desde)}),
    )


def accion_abrir(puerta: str, habitacion: str) -> Accion:
    return Accion(
        nombre=f"Abrir({puerta}, desde {habitacion})",
        precondiciones=frozenset({("en", "robot", habitacion), ("cerrada", puerta)}),
        add_list=frozenset({("abierta", puerta)}),
        del_list=frozenset({("cerrada", puerta)}),
    )


def accion_tomar(objeto: str, habitacion: str) -> Accion:
    return Accion(
        nombre=f"Tomar({objeto}, en {habitacion})",
        precondiciones=frozenset({("en", "robot", habitacion), ("en", objeto, habitacion), ("manos_libres", "robot")}),
        add_list=frozenset({("sosteniendo", "robot", objeto)}),
        del_list=frozenset({("en", objeto, habitacion), ("manos_libres", "robot")}),
    )


def accion_dejar(objeto: str, habitacion: str) -> Accion:
    return Accion(
        nombre=f"Dejar({objeto}, en {habitacion})",
        precondiciones=frozenset({("en", "robot", habitacion), ("sosteniendo", "robot", objeto)}),
        add_list=frozenset({("en", objeto, habitacion), ("manos_libres", "robot")}),
        del_list=frozenset({("sosteniendo", "robot", objeto)}),
    )


def generar_operadores(conexiones: list[tuple[str, str, str]], objetos: list[str], habitaciones: list[str]) -> list[Accion]:
    operadores = []
    for puerta, hab_a, hab_b in conexiones:
        operadores.append(accion_mover(hab_a, hab_b, puerta))
        operadores.append(accion_mover(hab_b, hab_a, puerta))
        operadores.append(accion_abrir(puerta, hab_a))
        operadores.append(accion_abrir(puerta, hab_b))
    for objeto in objetos:
        for habitacion in habitaciones:
            operadores.append(accion_tomar(objeto, habitacion))
            operadores.append(accion_dejar(objeto, habitacion))
    return operadores

## Construcción del mapa y del estado inicial

Se instancia el mapa descrito arriba (`conexiones`) y se generan todos los operadores del dominio. Los hechos `conecta` se agregan en ambos sentidos porque una puerta conecta dos habitaciones simétricamente: si `p1` conecta `cocina` con `pasillo`, el robot puede moverse en cualquiera de los dos sentidos por ella.

In [13]:
habitaciones = ["cocina", "pasillo", "sala", "comedor", "recamara"]
conexiones = [
    ("p1", "cocina", "pasillo"),
    ("p2", "pasillo", "sala"),
    ("p3", "pasillo", "comedor"),
    ("p4", "sala", "recamara"),
]
puertas = ["p1", "p2", "p3", "p4"]
objetos = ["caja"]

operadores = generar_operadores(conexiones, objetos, habitaciones)

hechos_conexion = set()
for puerta, hab_a, hab_b in conexiones:
    hechos_conexion.add(("conecta", puerta, hab_a, hab_b))
    hechos_conexion.add(("conecta", puerta, hab_b, hab_a))

estado_inicial = frozenset(
    hechos_conexion
    | {
        ("en", "robot", "cocina"),
        ("en", "caja", "comedor"),
        ("manos_libres", "robot"),
        ("abierta", "p1"),
        ("cerrada", "p2"),
        ("abierta", "p3"),
        ("cerrada", "p4"),
    }
)

print(f"Operadores instanciados: {len(operadores)}")
print(f"Hechos en el estado inicial: {len(estado_inicial)}")

Operadores instanciados: 26
Hechos en el estado inicial: 15


## El planificador: búsqueda en el espacio de estados

`planificar` implementa una búsqueda en anchura (BFS) sobre el espacio de estados: parte del estado inicial y, en cada paso, expande **todas** las acciones aplicables (aquellas cuyas precondiciones son subconjunto del estado actual), generando el estado sucesor con `aplicar`. Se detiene en cuanto encuentra un estado que satisface la meta (es decir, cuando `meta` es subconjunto del estado) y devuelve la secuencia de acciones que llevó hasta ahí.

Se eligió BFS y no una búsqueda en profundidad porque, al explorar los estados en orden de número de acciones ejecutadas, **garantiza el plan más corto** posible — una propiedad deseable en un dominio como este, donde no hay costos distintos por acción y el espacio de estados es pequeño. El conjunto `visitados` evita reprocesar un mismo estado por dos caminos distintos, lo cual es clave porque el espacio de estados de este dominio tiene ciclos (p. ej. moverse y devolverse). Si la frontera se vacía sin haber alcanzado la meta, el problema no tiene solución con los operadores disponibles y la función devuelve `None`.

In [14]:
def planificar(estado_inicial: frozenset, meta: frozenset, operadores: list[Accion]) -> list[Accion] | None:
    frontera = deque([(estado_inicial, [])])
    visitados = {estado_inicial}
    while frontera:
        estado, plan = frontera.popleft()
        if meta <= estado:
            return plan
        for operador in operadores:
            if operador.aplicable(estado):
                nuevo_estado = operador.aplicar(estado)
                if nuevo_estado not in visitados:
                    visitados.add(nuevo_estado)
                    frontera.append((nuevo_estado, plan + [operador]))
    return None

## Mecánica interna de `planificar`: por qué garantiza el plan más corto

Algunos detalles de la implementación que no son evidentes a simple vista:

- **`frontera` es un `deque`, no una `list`.** Cada elemento es el par `(estado, plan_para_llegar_ahí)`: el plan viaja pegado al estado, así que no hace falta reconstruir el camino al final con punteros hacia atrás. `popleft()` es O(1) en un `deque`; en una `list` sería O(n) porque desplaza todos los elementos — con más estados, esto importa.
- **`plan + [operador]` crea una lista nueva en cada rama**, en vez de modificar `plan` con `.append()`. Es necesario porque un mismo estado puede generar varios estados hijos, y cada rama necesita su propia copia del camino recorrido sin pisar las demás.
- **`visitados` (un `set` de `frozenset`) poda el árbol de búsqueda.** Sin él, el algoritmo re-expandiría estados ya vistos indefinidamente — el dominio tiene ciclos triviales (moverse y volver) — degenerando en un bucle que nunca termina o que explora un número explosivo de estados repetidos.
- **Por qué BFS garantiza el plan más corto:** BFS visita los estados en orden estrictamente creciente del número de acciones necesarias para llegarles (todos los de 0 acciones, luego todos los de 1, luego todos los de 2...). La primera vez que aparece en la frontera un estado que satisface la meta, es imposible que exista un camino más corto todavía sin descubrir — si existiera, ya habría sido desencolado en una iteración anterior.
- **Terminación:** el espacio de estados alcanzables desde `estado_inicial` es finito (hay un número finito de hechos posibles combinando habitaciones, puertas y objetos). Si la frontera se vacía sin encontrar la meta, es porque se agotó ese espacio finito por completo, y devolver `None` es la respuesta correcta: la meta es irrealizable con los operadores disponibles.

## Ejemplo reducido, trazado paso a paso

Para poder seguir la ejecución de `planificar` con el detalle de cada iteración, se arma un dominio reducido con solo 3 habitaciones y 2 puertas (reutilizando `accion_mover` y `accion_abrir`, ya definidas arriba):

`cocina —p1 (abierta)— pasillo —p2 (cerrada)— sala`

Estado inicial: robot en `cocina`. Meta: `en(robot, sala)`. Como `p2` está cerrada, el robot no puede llegar directo — el planificador tiene que descubrir por sí mismo, a partir de las precondiciones, que hace falta abrir `p2` antes de cruzar.

In [15]:
habitaciones_demo = ["cocina", "pasillo", "sala"]
conexiones_demo = [("p1", "cocina", "pasillo"), ("p2", "pasillo", "sala")]

operadores_demo = []
for puerta, hab_a, hab_b in conexiones_demo:
    operadores_demo.append(accion_mover(hab_a, hab_b, puerta))
    operadores_demo.append(accion_mover(hab_b, hab_a, puerta))
    operadores_demo.append(accion_abrir(puerta, hab_a))
    operadores_demo.append(accion_abrir(puerta, hab_b))

hechos_conexion_demo = set()
for puerta, hab_a, hab_b in conexiones_demo:
    hechos_conexion_demo.add(("conecta", puerta, hab_a, hab_b))
    hechos_conexion_demo.add(("conecta", puerta, hab_b, hab_a))

estado_inicial_demo = frozenset(
    hechos_conexion_demo | {("en", "robot", "cocina"), ("abierta", "p1"), ("cerrada", "p2")}
)
meta_demo = frozenset({("en", "robot", "sala")})

print(f"tipo de estado_inicial_demo: {type(estado_inicial_demo)}")
print(f"estado_inicial_demo ({len(estado_inicial_demo)} hechos):")
print(estado_inicial_demo)
print()
print(f"tipo de meta_demo: {type(meta_demo)}")
print(f"meta_demo:")
print(meta_demo)

tipo de estado_inicial_demo: <class 'frozenset'>
estado_inicial_demo (7 hechos):
frozenset({('conecta', 'p1', 'pasillo', 'cocina'), ('conecta', 'p2', 'pasillo', 'sala'), ('abierta', 'p1'), ('en', 'robot', 'cocina'), ('cerrada', 'p2'), ('conecta', 'p2', 'sala', 'pasillo'), ('conecta', 'p1', 'cocina', 'pasillo')})

tipo de meta_demo: <class 'frozenset'>
meta_demo:
frozenset({('en', 'robot', 'sala')})


In [16]:
def planificar_trazado(estado_inicial, meta, operadores):
    """Misma lógica que planificar(), pero imprime cada iteración de BFS."""
    frontera = deque([(estado_inicial, [])])
    visitados = {estado_inicial}
    iteracion = 0
    while frontera:
        iteracion += 1
        estado, plan = frontera.popleft()
        camino = " -> ".join(a.nombre for a in plan) or "(estado inicial, sin acciones)"
        print(f"Iteración {iteracion}: se desencola el estado alcanzado con: {camino}")
        if meta <= estado:
            print(f"  -> La meta ya es subconjunto de este estado. Plan devuelto ({len(plan)} acción(es)).\n")
            return plan
        aplicables = [op for op in operadores if op.aplicable(estado)]
        print(f"  Operadores aplicables en este estado: {len(aplicables)}")
        for operador in aplicables:
            nuevo_estado = operador.aplicar(estado)
            if nuevo_estado not in visitados:
                visitados.add(nuevo_estado)
                frontera.append((nuevo_estado, plan + [operador]))
                print(f"    + se agrega a la frontera -> {operador.nombre}")
            else:
                print(f"    - descartado, estado ya visitado -> {operador.nombre}")
        print()
    print("Frontera agotada sin encontrar la meta.")
    return None


plan_demo = planificar_trazado(estado_inicial_demo, meta_demo, operadores_demo)
print("Plan final:")
for paso, accion in enumerate(plan_demo, start=1):
    print(f"{paso}. {accion.nombre}")

Iteración 1: se desencola el estado alcanzado con: (estado inicial, sin acciones)
  Operadores aplicables en este estado: 1
    + se agrega a la frontera -> Mover(robot, cocina -> pasillo, por p1)

Iteración 2: se desencola el estado alcanzado con: Mover(robot, cocina -> pasillo, por p1)
  Operadores aplicables en este estado: 2
    - descartado, estado ya visitado -> Mover(robot, pasillo -> cocina, por p1)
    + se agrega a la frontera -> Abrir(p2, desde pasillo)

Iteración 3: se desencola el estado alcanzado con: Mover(robot, cocina -> pasillo, por p1) -> Abrir(p2, desde pasillo)
  Operadores aplicables en este estado: 2
    + se agrega a la frontera -> Mover(robot, pasillo -> cocina, por p1)
    + se agrega a la frontera -> Mover(robot, pasillo -> sala, por p2)

Iteración 4: se desencola el estado alcanzado con: Mover(robot, cocina -> pasillo, por p1) -> Abrir(p2, desde pasillo) -> Mover(robot, pasillo -> cocina, por p1)
  Operadores aplicables en este estado: 1
    - descartado, es

### Lectura de la traza

- En la primera iteración solo hay un operador aplicable: `Mover(cocina→pasillo,p1)` (p1 ya está abierta). El resto —incluidos ambos `Abrir`— falla porque sus precondiciones no se cumplen (p1 no está cerrada, o el robot no está en la habitación correcta).
- En la segunda iteración, `Mover(pasillo→cocina,p1)` sí es aplicable, pero el estado que genera es **idéntico** al estado inicial (nada más cambió), así que ya está en `visitados` y se descarta — así es como se poda el ciclo trivial "ir y volver" sin que el algoritmo lo vuelva a explorar.
- `Mover(pasillo→sala,p2)` aparece como **no aplicable** hasta que, en una iteración anterior, `Abrir(p2,pasillo)` se ejecutó y generó un estado donde `p2` ya está abierta. El planificador nunca recibió la instrucción "abre la puerta antes de cruzar" — lo dedujo únicamente de que la precondición `abierta(p2)` no se cumplía todavía.
- El plan final de 3 acciones coincide exactamente con el que produce el sistema completo para el caso de prueba "Llegar a la sala (requiere abrir p2)", lo cual confirma que este ejemplo reducido reproduce fielmente el comportamiento del planificador real con un espacio de estados lo bastante pequeño para seguirlo iteración por iteración.

## Demostración: el robot lleva la caja a la recámara

La meta es que el robot termine en la recámara y que la caja también esté ahí: `en(robot, recamara)` y `en(caja, recamara)`. Para lograrlo el planificador tiene que encadenar movimientos, apertura de puertas y la toma/entrega del objeto — ninguna acción por sí sola lo resuelve.

In [17]:
meta_entrega = frozenset({("en", "robot", "recamara"), ("en", "caja", "recamara")})

plan_entrega = planificar(estado_inicial, meta_entrega, operadores)

print(f"Plan encontrado ({len(plan_entrega)} acciones):\n")
for paso, accion in enumerate(plan_entrega, start=1):
    print(f"{paso}. {accion.nombre}")

Plan encontrado (9 acciones):

1. Mover(robot, cocina -> pasillo, por p1)
2. Abrir(p2, desde pasillo)
3. Mover(robot, pasillo -> comedor, por p3)
4. Tomar(caja, en comedor)
5. Mover(robot, comedor -> pasillo, por p3)
6. Mover(robot, pasillo -> sala, por p2)
7. Abrir(p4, desde sala)
8. Mover(robot, sala -> recamara, por p4)
9. Dejar(caja, en recamara)


## Trazado del estado paso a paso

Para verificar que el plan es correcto y no solo que "se encontró algo", se re-ejecuta la secuencia desde el estado inicial comprobando en cada paso que la acción sigue siendo aplicable (sus precondiciones se cumplen en el estado alcanzado hasta ese punto) y mostrando cómo evoluciona la posición del robot y de la caja.

In [18]:
estado_actual = estado_inicial
print("Estado inicial -> robot en cocina, caja en comedor\n")
for paso, accion in enumerate(plan_entrega, start=1):
    assert accion.aplicable(estado_actual), f"Precondiciones no satisfechas en el paso {paso}"
    estado_actual = accion.aplicar(estado_actual)
    ubicacion_robot = next(h[2] for h in estado_actual if len(h) == 3 and h[0] == "en" and h[1] == "robot")
    if ("sosteniendo", "robot", "caja") in estado_actual:
        estado_caja = "en manos del robot"
    else:
        estado_caja = next(h[2] for h in estado_actual if len(h) == 3 and h[0] == "en" and h[1] == "caja")
    print(f"{paso}. {accion.nombre:<38} -> robot en {ubicacion_robot:<10} | caja: {estado_caja}")

print("\n¿Meta satisfecha al final del plan?", meta_entrega <= estado_actual)

Estado inicial -> robot en cocina, caja en comedor

1. Mover(robot, cocina -> pasillo, por p1) -> robot en pasillo    | caja: comedor
2. Abrir(p2, desde pasillo)               -> robot en pasillo    | caja: comedor
3. Mover(robot, pasillo -> comedor, por p3) -> robot en comedor    | caja: comedor
4. Tomar(caja, en comedor)                -> robot en comedor    | caja: en manos del robot
5. Mover(robot, comedor -> pasillo, por p3) -> robot en pasillo    | caja: en manos del robot
6. Mover(robot, pasillo -> sala, por p2)  -> robot en sala       | caja: en manos del robot
7. Abrir(p4, desde sala)                  -> robot en sala       | caja: en manos del robot
8. Mover(robot, sala -> recamara, por p4) -> robot en recamara   | caja: en manos del robot
9. Dejar(caja, en recamara)               -> robot en recamara   | caja: recamara

¿Meta satisfecha al final del plan? True


# Prueba del sistema

La batería cubre tres casos adicionales al de la demostración, para comprobar que el planificador se comporta correctamente tanto en metas alcanzables (simples y complejas) como en una meta **imposible**:

1. **Meta simple sin obstáculos**: el robot solo necesita llegar a una habitación cercana con la puerta ya abierta.
2. **Meta que requiere abrir una puerta**: el robot debe llegar a una habitación cuya puerta de acceso está cerrada.
3. **Meta completa de entrega** (la misma de la demostración), para confirmar que el resultado es reproducible.
4. **Meta inalcanzable**: se pide llevar al robot a una habitación que no existe en el mapa (`jardin`), para comprobar que el planificador termina y devuelve `None` en vez de quedarse buscando indefinidamente o fallar con un error.

In [19]:
casos_prueba = [
    ("Llegar al pasillo (puerta ya abierta)", frozenset({("en", "robot", "pasillo")})),
    ("Llegar a la sala (requiere abrir p2)", frozenset({("en", "robot", "sala")})),
    ("Entregar la caja en la recamara", meta_entrega),
    ("Llegar a un jardín inexistente", frozenset({("en", "robot", "jardín")})),
]

for descripcion, meta in casos_prueba:
    plan = planificar(estado_inicial, meta, operadores)
    if plan is None:
        print(f"{descripcion}\n  -> Sin plan (meta inalcanzable con los operadores disponibles)\n")
    else:
        print(f"{descripcion}\n  -> Plan de {len(plan)} accion(es):")
        for paso, accion in enumerate(plan, start=1):
            print(f"     {paso}. {accion.nombre}")
        print()

Llegar al pasillo (puerta ya abierta)
  -> Plan de 1 accion(es):
     1. Mover(robot, cocina -> pasillo, por p1)

Llegar a la sala (requiere abrir p2)
  -> Plan de 3 accion(es):
     1. Mover(robot, cocina -> pasillo, por p1)
     2. Abrir(p2, desde pasillo)
     3. Mover(robot, pasillo -> sala, por p2)

Entregar la caja en la recamara
  -> Plan de 9 accion(es):
     1. Mover(robot, cocina -> pasillo, por p1)
     2. Abrir(p2, desde pasillo)
     3. Mover(robot, pasillo -> comedor, por p3)
     4. Tomar(caja, en comedor)
     5. Mover(robot, comedor -> pasillo, por p3)
     6. Mover(robot, pasillo -> sala, por p2)
     7. Abrir(p4, desde sala)
     8. Mover(robot, sala -> recamara, por p4)
     9. Dejar(caja, en recamara)

Llegar a un jardín inexistente
  -> Sin plan (meta inalcanzable con los operadores disponibles)



## Lectura de resultados

- **Llegar al pasillo** se resuelve con una sola acción (`Mover`), porque `p1` ya está abierta en el estado inicial: el planificador no agrega ningún paso innecesario, lo que confirma que la búsqueda en anchura efectivamente encuentra el plan más corto.
- **Llegar a la sala** obliga a abrir `p2` antes de poder cruzarla, y el plan de 3 acciones lo muestra explícitamente (`Mover` a pasillo, `Abrir p2`, `Mover` a sala): el sistema descubre por sí mismo, a partir de las precondiciones, que falta abrir la puerta — nadie se lo indicó directamente.
- **Entregar la caja en la recámara** reproduce el mismo plan de 9 acciones que la demostración: el orden exacto entre acciones independientes (por ejemplo, abrir `p2` antes o después de recoger la caja) puede variar entre ejecuciones porque ambas son válidas y de igual costo, pero el número de acciones y el resultado final son siempre los mismos.
- **El jardín inexistente** confirma el caso negativo: como ninguna habitación del mapa se llama `jardin` y ningún operador puede producir el hecho `en(robot, jardin)`, la búsqueda agota todo el espacio de estados alcanzables y el planificador responde `None` de forma controlada, en vez de entrar en un ciclo infinito — importante porque en un sistema real este es el aviso de que la meta pedida es, sencillamente, imposible con las acciones disponibles.

# Conclusiones

El sistema implementado sigue el modelo STRIPS de forma directa: hechos como tuplas, estados como conjuntos, y acciones como precondición/lista-añadir/lista-borrar, tal como lo definieron Fikes y Nilsson (1971). Sobre esa representación, una búsqueda en anchura genérica —que no sabe nada de robots, habitaciones ni puertas— fue suficiente para resolver tanto movimientos simples como una tarea compuesta de varios pasos interdependientes (moverse, abrir puertas, tomar y dejar un objeto), descubriendo el orden correcto de las acciones únicamente a partir de sus precondiciones.

La ventaja de este enfoque es la misma que la de los sistemas basados en reglas de las prácticas anteriores: es **declarativo**. Para agregar una habitación, una puerta o un tipo de acción nuevo no hace falta tocar el planificador, solo describir sus precondiciones y efectos — el comportamiento inteligente (encontrar el plan) emerge de la búsqueda, no de lógica escrita a mano para el caso particular.

La limitación es la esperable en STRIPS clásico: el dominio es completamente determinista (una acción siempre produce el mismo resultado) y de información completa (el estado se conoce por completo), y la búsqueda en anchura sin heurística —adecuada aquí por lo pequeño del espacio de estados— dejaría de ser práctica en un dominio con muchas más habitaciones, objetos y robots, donde haría falta una heurística (p. ej. distancia al objetivo) o un algoritmo más eficiente como Graphplan o un planificador basado en SAT.

## Referencias

Fikes, R. E., & Nilsson, N. J. (1971). STRIPS: A new approach to the application of theorem proving to problem solving. *Artificial Intelligence*, 2(3-4), 189-208.

Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.

Ghallab, M., Nau, D., & Traverso, P. (2004). *Automated Planning: Theory and Practice*. Morgan Kaufmann.

Hernández Parada, M. S. (2026). *Maestría en Inteligencia Artificial — Prácticas* [Repositorio de GitHub]. GitHub. [https://github.com/RKCbas/Maestria-en-Inteligencia-Artificial---Practicas/blob/main/Cuatrimestre%203/7%20-%20Razonamiento%20Inteligente/Practica%203.1/Implementar%20un%20sistema%20STRIPS%20b%C3%A1sico%20para%20proporcionar%20movimiento%20a%20un%20robot.ipynb](https://github.com/RKCbas/Maestria-en-Inteligencia-Artificial---Practicas/blob/main/Cuatrimestre%203/7%20-%20Razonamiento%20Inteligente/Practica%203.1/Implementar%20un%20sistema%20STRIPS%20b%C3%A1sico%20para%20proporcionar%20movimiento%20a%20un%20robot.ipynb)